# Agricultural PDF reports — extraction, cleaning & QA

**Source:** 10 annual Ukrainian agricultural statistical reports (forms **29-сг** and **50-сг**, 2021–2025).  
**Goal:** reproducibly create analysis-ready datasets, with traceability to source PDF pages and without inventing missing values.

This notebook:
1. Reads `data_pdf.zip` (or a folder with the extracted PDF files).
2. Recognizes report **year** and **form** from PDF contents.
3. Extracts all tables into **auditable row-level text + cell records**.
4. Extracts an initial structured crop production panel for the stable 29-сг 1.1/1.2 sections.
5. Preserves 50-сг tables for review in `raw_report_rows.csv` and `report_row_cells.csv` rather than pretending the distinct cost/inventory layouts are interchangeable.
6. Writes quality reports and flags questionable records for manual checking.

**Privacy:** These are reports of an identifiable enterprise. Do not commit raw PDFs, identifiable source extracts or derived enterprise-level datasets to a public GitHub repository without permission.


## 0. Install dependencies

Run once in Colab/Jupyter. In your project environment you can instead install from `requirements.txt`.

In [1]:
%pip -q install "pymupdf>=1.24" "pandas>=2.0" "numpy>=1.26" "openpyxl>=3.1"


[notice] A new release of pip is available: 26.0.1 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 1. Configure paths

**Recommended layout:** put `data_pdf.zip` at `data/agriculture/raw/data_pdf.zip` relative to the repository root, or upload it to Colab. Files are read **recursively** from the archive and Apple `__MACOSX` metadata is ignored.

By default, generated files go into `data/agriculture/processed/`.

In [2]:
from pathlib import Path
import io, re, json, zipfile, unicodedata
from collections import Counter
import fitz  # PyMuPDF
import pandas as pd
import numpy as np
from IPython.display import display

# Auto-detect whether this notebook is started from the repository root or notebooks/
CWD = Path.cwd()
ROOT = CWD.parent if CWD.name == "notebooks" else CWD
CANDIDATES = [
    ROOT / "data" / "agriculture" / "raw" / "data_pdf.zip",
    CWD / "data_pdf.zip",
    ROOT / "data_pdf.zip",
    Path("/content/data_pdf.zip"),
]
ZIP_PATH = next((p for p in CANDIDATES if p.exists()), None)
if ZIP_PATH is None:
    raise FileNotFoundError(
        "Place data_pdf.zip in data/agriculture/raw/ or upload it to the current directory / Colab."
    )
OUT = ROOT / "data" / "agriculture" / "processed"
OUT.mkdir(parents=True, exist_ok=True)
print("Input:", ZIP_PATH)
print("Output:", OUT.resolve())

/Users/margopaduchak/Library/Python/3.12/lib/python/site-packages/fitz/__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


Input: /Users/margopaduchak/Desktop/прогнози/forecasting-and-decision-optimization/data/agriculture/raw/data_pdf.zip
Output: /Users/margopaduchak/Desktop/прогнози/forecasting-and-decision-optimization/data/agriculture/processed


## 2. Load and classify documents

The year is identified from the **document's contents** rather than inferred from the filename (some filenames contain submission dates / report identifiers).

In [3]:
def normalize_text(s):
    return unicodedata.normalize("NFKC", str(s)).replace("\xa0", " ").replace("\u2009", " ")

def classify_pdf(first_page, name):
    text = normalize_text(first_page)
    form = ("29-сг" if re.search(r"29\s*[-–]?\s*сг", text, re.I)
            else "50-сг" if re.search(r"50\s*[-–]?\s*сг", text, re.I)
            else "29-сг" if "S27029" in name else "50-сг" if "S27021" in name else None)
    # The title pages contain an explicit «за 2025 рік» etc.
    m = re.search(r"(?:за|За)\s+(20\d{2})\s*(?:рік|року|р\.)", text)
    if not m:
        m = re.search(r"\b(202[1-5])\b", text[:3500])
    return form, int(m.group(1)) if m else None

reports = []
with zipfile.ZipFile(ZIP_PATH) as archive:
    names = [x for x in archive.namelist()
             if x.lower().endswith(".pdf") and not x.startswith("__MACOSX/")]
    for name in names:
        data = archive.read(name)
        try:
            doc = fitz.open(stream=data, filetype="pdf")
            page0 = doc[0].get_text(sort=True)
            form, year = classify_pdf(page0, name)
            reports.append({"filename": name.split("/")[-1], "form": form,
                            "year": year, "pages": len(doc), "data": data})
            doc.close()
        except Exception as exc:
            raise ValueError(f"Cannot read PDF: {name}: {exc}") from exc
metadata = pd.DataFrame([{k:v for k,v in r.items() if k != "data"} for r in reports])
display(metadata.sort_values(["form", "year"]))
expected = {(f, y) for f in ("29-сг", "50-сг") for y in range(2021, 2026)}
found = {(r["form"], r["year"]) for r in reports}
assert expected == found, f"Missing: {expected-found}; unexpected: {found-expected}"
assert len(reports) == 10, f"Expected 10 reports, got {len(reports)}"
metadata.to_csv(OUT / "source_manifest.csv", index=False)
print("Validated: 5 years × 2 forms = 10 PDFs")

,filename,form,year,pages
1,610020003782954S270291410000001122021.pdf,29-сг,2021,10
2,610020003782954S270291510000001122022.pdf,29-сг,2022,10
7,S2702916_29_╤ü╨│_╤Ç╤û╤ç╨╜╨░_╨ù╨▓╤û╤é_╨┐╤Ç╨╛_╨┐...,29-сг,2023,12
0,S2702917_Γäû_29_╤ü╨│_╤Ç╤û╤ç╨╜╨░_╨ù╨▓╤û╤é_╨┐╤Ç╨...,29-сг,2024,12
3,S2702918_Γäû_29_╤ü╨│_╤Ç╤û╤ç╨╜╨░_╨ù╨▓╤û╤é_╨┐╤Ç╨...,29-сг,2025,12
5,610020003782954S270211610000002122021.pdf,50-сг,2021,8
9,610020003782954S270211710000003122022.pdf,50-сг,2022,7
4,S2702117_50_╤ü╨│_╤Ç╤û╤ç╨╜╨░_╨ù╨▓╤û╤é_╨┐╤Ç╨╛_╨╛...,50-сг,2023,10
6,S2702118_Γäû_50_╤ü╨│_╤Ç╤û╤ç╨╜╨░_╨ù╨▓╤û╤é_╨┐╤Ç╨...,50-сг,2024,10
8,S2702118_Γäû_50_╤ü╨│_╤Ç╤û╤ç╨╜╨░_╨ù╨▓╤û╤é_╨┐╤Ç╨...,50-сг,2025,10


Validated: 5 years × 2 forms = 10 PDFs


## 3. Extract row codes, labels and cells from PDF pages

We use **word geometry** instead of copying `page.get_text()` as a giant string. This preserves the PDF page number, row identifier, and original cell text. We deliberately keep a raw audit layer because column layouts differ across forms and years.

`row_code` is the four-digit statistical form code; only a plausible left-side column is recognized, which helps avoid identifying numerical values as row codes. **Manual validation is still necessary before using raw 50-сг columns in optimization.**

In [4]:
def clean_spaces(s):
    return re.sub(r"\s+", " ", normalize_text(s)).strip()

def words_on_line(words, y, tol=3.0):
    return sorted([w for w in words if abs(w[1]-y) < tol], key=lambda w:w[0])

def join_word_fragments(words, max_gap=9):
    if not words: return ""
    output = words[0][4]
    for prev, curr in zip(words, words[1:]):
        output += ("" if curr[0]-prev[2] <= max_gap else " ") + curr[4]
    return clean_spaces(output)

def row_candidates(page, form):
    words = page.get_text("words")
    # Candidate row-code positions vary by form, year, and page layout.
    # A code in a line with some left-hand label is more likely to be a table row.
    width = page.rect.width
    for w in words:
        token = w[4]
        if not re.fullmatch(r"\d{4}", token): continue
        if not ((0.27*width < w[0] < 0.39*width) if form == "29-сг" else (0.13*width < w[0] < 0.39*width)): continue
        nearby = words_on_line(words, w[1], tol=2.9)
        left = [x for x in nearby if x[2] <= w[0] and x != w]
        right = [x for x in nearby if x[0] >= w[2]]
        label = join_word_fragments(left, max_gap=3)
        if not label or not right: continue
        if not re.search(r"[А-Яа-яІіЇїЄєҐґA-Za-z]", label): continue
        # Sort cells geometrically; keep text and coordinates for auditing.
        numeric_right = [x for x in right if re.fullmatch(r"[0-9][0-9.,]*|[-–—хxХX]", x[4])]
        if len(numeric_right) < 2: continue
        yield {"row_code": token, "label_raw": label,
               "values_raw": join_word_fragments(right),
               "words_right": right, "code_x": w[0], "code_y":w[1]}

raw_rows, raw_cells = [], []
for report in reports:
    doc = fitz.open(stream=report["data"], filetype="pdf")
    for page_index, page in enumerate(doc):
        page_text = page.get_text(sort=True)
        first_lines = [clean_spaces(x) for x in page_text.splitlines() if x.strip()]
        page_heading = " | ".join(first_lines[:4])[:350]
        for candidate in row_candidates(page, report["form"]):
            row_id = len(raw_rows)
            raw_rows.append({"row_id":row_id, "form":report["form"],
                "year":report["year"], "source_file":report["filename"],
                "page":page_index+1, "heading":page_heading,
                "row_code":candidate["row_code"], "label_raw":candidate["label_raw"],
                "values_raw":candidate["values_raw"],
                "code_x":round(candidate["code_x"],1), "code_y":round(candidate["code_y"],1)})
            for index, w in enumerate(candidate["words_right"]):
                raw_cells.append({"row_id":row_id, "token_index":index,
                                  "x0":round(w[0],1), "x1":round(w[2],1),
                                  "text":w[4]})
    doc.close()
raw_df = pd.DataFrame(raw_rows)
cell_df = pd.DataFrame(raw_cells)
raw_df.to_csv(OUT / "raw_report_rows.csv", index=False)
cell_df.to_csv(OUT / "report_row_cells.csv", index=False)
print("Candidate table rows:",len(raw_df),"| word-level cell fragments:",len(cell_df))
display(raw_df.groupby(["form","year"]).size().unstack(0))
display(raw_df[["form","year","page","row_code","label_raw","values_raw"]].head(12))

Candidate table rows: 1947 | word-level cell fragments: 14178


form,29-сг,50-сг
year,,
2021,251,122
2022,241,125
2023,261,139
2024,261,142
2025,261,144


,form,year,page,row_code,label_raw,values_raw
0,29-сг,2024,2,0030,(сумакодівіз0050до0270),"2680,00 2680,00 260836,00 234302,00 - -"
1,29-сг,2024,2,0050,Пшеницяозима,"600,00 600,00 59735,00 56968,00 - -"
2,29-сг,2024,2,0130,Ячміньозимий,- - - - - -
3,29-сг,2024,2,0160,Житоозиме,- - - - - -
4,29-сг,2024,2,0190,Тритикалеозиме,- - - - - -
5,29-сг,2024,2,0080,Пшеницяяра,- - - - - -
6,29-сг,2024,2,0140,Ячміньярий,"1560,00 1560,00 91361,00 85580,00 - -"
7,29-сг,2024,2,0170,Житояре,- - - - - -
8,29-сг,2024,2,0200,Тритикалеяре,- - - - - -
9,29-сг,2024,2,0110,Кукурудза,"520,00 520,00 109740,00 91754,00 - -"


## 4. Parse **29-сг** crop production (Sections 1.1 and 1.2)

Here the first four numeric columns consistently represent **sown area (ha), harvested area (ha), initial mass (centners), and post-processing mass (centners)**. We extract 2021–2025, while saving the original PDF reference for every observation.

`-` and `х` are kept as **missing values**, not silently replaced by zero. One centner = **0.1 metric tonnes**. We only output crop codes that are present; aggregate lines are flagged, not summed with their sub-crops. This stage deliberately excludes vegetable/perennial sections with different column meanings.

In [5]:
def combine_nearby_numeric_fragments(words, min_x, max_gap=10):
    """Return numeric tokens after code, combining thousands fragments by geometry."""
    vals = [w for w in words if w[0] > min_x and re.fullmatch(r"[0-9][0-9.,]*|[-–—хxХX]", w[4])]
    vals.sort(key=lambda x:x[0])
    result=[]
    for w in vals:
        txt=w[4]
        if result and w[0]-result[-1][1] < max_gap and re.fullmatch(r"[0-9][0-9.,]*",txt) and re.fullmatch(r"[0-9][0-9., ]*",result[-1][2]):
            result[-1]=(result[-1][0],w[2],result[-1][2]+" "+txt)
        else:
            result.append((w[0],w[2],txt))
    return [x[2] for x in result]

def ua_number(value):
    s=clean_spaces(value)
    if s in ("", "-", "–", "—", "х", "Х", "x", "X"):return np.nan
    s=s.replace(" ","").replace("'","").replace(",",".")
    try:return float(s)
    except ValueError:return np.nan

records=[]
for row in raw_rows:
    if row["form"] != "29-сг":continue
    # The grain and technical crops sections are on pages 2–4 of all supplied 29-сг reports.
    # These tables share four leading production columns; other sections do not.
    if not (2 <= row["page"] <= (3 if row["year"] <= 2022 else 4)): continue
    tokens = cell_df.loc[cell_df.row_id==row["row_id"]].sort_values("x0")
    # Text of the raw cell words, no unverified decoding of cost/inventory columns.
    input_words = [(float(t.x0),0,float(t.x1),0,str(t.text)) for t in tokens.itertuples()]
    parts = combine_nearby_numeric_fragments(input_words, row["code_x"]+10)
    if len(parts) < 4:continue
    # Reject rows that look like table instructions rather than crop measurements.
    label = clean_spaces(row["label_raw"])
    if not re.search(r"[А-Яа-яІіЇїЄєҐґ]",label):continue
    values = list(map(ua_number,parts[:4]))
    records.append({"year":row["year"], "crop_code":row["row_code"],
        "crop_name_raw":label,"sown_area_ha":values[0],
        "harvested_area_ha":values[1],"harvest_initial_centners":values[2],
        "harvest_after_processing_centners":values[3],
        "harvest_tonnes":values[3]/10.0 if pd.notna(values[3]) else np.nan,
        "yield_tonnes_per_ha":values[3]/10.0/values[1] if pd.notna(values[3]) and pd.notna(values[1]) and values[1]>0 else np.nan,
        "source_file":row["source_file"], "source_page":row["page"],
        "raw_row_id":row["row_id"], "parsed_tokens":json.dumps(parts,ensure_ascii=False)})
production = pd.DataFrame(records)
production = production.sort_values(["year","crop_code"]).reset_index(drop=True)
# Some page layouts include rows that are table headings: preserve and flag duplicates for QA.
production["potential_aggregate"] = production.crop_name_raw.str.contains(r"усього|всього|сума|культури зернові|культури технічні", case=False,regex=True)
production.to_csv(OUT / "crop_production.csv",index=False)
print("Production rows:",len(production))
display(production.groupby("year").agg(rows=("crop_code","count"),harvest_reported=("harvest_tonnes","count")))
display(production.loc[production.harvest_tonnes.gt(0),["year","crop_code","crop_name_raw","harvested_area_ha","harvest_tonnes","yield_tonnes_per_ha"]].head(25))

Production rows: 475


,rows,harvest_reported
year,,
2021,93,3
2022,91,4
2023,97,4
2024,97,4
2025,97,4


,year,crop_code,crop_name_raw,harvested_area_ha,harvest_tonnes,yield_tonnes_per_ha
0,2021,0050,Пшеницяозима,1100.0,8459.3,7.690273
2,2021,0110,Кукурудза,1600.0,19368.8,12.105500
4,2021,0140,Ячмiньярий,720.0,3784.5,5.256250
93,2022,0030,тазернобобові,2900.0,20346.1,7.015897
94,2022,0050,Пшеницяозима,1100.0,7203.5,6.548636
96,2022,0110,Кукурудза,700.0,6202.5,8.860714
98,2022,0140,Ячмiньярий,1100.0,6940.1,6.309182
184,2023,0030,(сумакодівіз0050до0270),3535.0,28972.9,8.196011
185,2023,0050,Пшеницяозима,1400.0,11116.2,7.940143
187,2023,0110,Кукурудза,640.0,9109.7,14.233906


## 5. Parse 50-сг sections with safe provenance

The 50-сг form has **different tables and column counts across report years**. This notebook exports:

- `cost_rows_for_review.csv` — crop cost structure (section 1.1), including original row values and source page;
- `inventory_rows_for_review.csv` — inventory movements (section 3), including original row values and source page;
- `raw_report_rows.csv` / `report_row_cells.csv` — geometry-preserving raw extraction.

**The files are review datasets, not validated numeric feature tables.** Interpreting the 50-сг values as specific numeric columns without verifying each version's header could silently corrupt your optimization. In the next step, we can map each form version's column coordinates explicitly and validate against the PDF.

In [6]:
def section_type(row):
    text = row["heading"].casefold()
    if row["form"] != "50-сг":return "other"
    # Form 50 crop cost section normally appears on p1/2 depending on the form version.
    if row["page"] <= 2 and (row["row_code"].startswith("11") or row["row_code"].startswith("12")):
        return "crop_costs_candidate"
    # Inventory movement section commonly contains the 3000-series crop rows.
    if row["row_code"].startswith("30") and 3 <= row["page"] <= 7:
        return "inventory_candidate"
    return "other"
raw_df["section_guess"] = raw_df.apply(section_type,axis=1)
cost_review = raw_df[raw_df.section_guess=="crop_costs_candidate"].copy()
inventory_review = raw_df[raw_df.section_guess=="inventory_candidate"].copy()
cost_review.to_csv(OUT / "cost_rows_for_review.csv",index=False)
inventory_review.to_csv(OUT / "inventory_rows_for_review.csv",index=False)
print("Cost rows for review:",len(cost_review), "| Inventory rows for review:",len(inventory_review))
display(raw_df.groupby(["form","year","section_guess"]).size().to_frame("rows").head(25))

Cost rows for review: 97 | Inventory rows for review: 184


rows
form  year section_guess             
29-сг 2021 other                  251
      2022 other                  241
      2023 other                  261
      2024 other                  261
      2025 other                  261
50-сг 2021 crop_costs_candidate    15
           inventory_candidate     33
           other                   74
      2022 crop_costs_candidate    16
           inventory_candidate     34
           other                   75
      2023 crop_costs_candidate    22
           inventory_candidate     39
           other                   78
      2024 crop_costs_candidate    22
           inventory_candidate     39
           other                   81
      2025 crop_costs_candidate    22
           inventory_candidate     39
           other                   83

## 6. Quality checks (do not suppress failures)

These checks are meant to surface parsing mistakes and implausible values. Missing data is **not** assumed to be zero. Source PDF row names can also vary between years, so `crop_code` is the preferred linkage key after verifying its meaning.

In [7]:
issues=[]
def add_issue(issue, subset, severity="warning"):
    for _,r in subset.iterrows():
        issues.append({"severity":severity,"issue":issue,"year":r.get("year"),
            "crop_code":r.get("crop_code"),"source_file":r.get("source_file"),
            "source_page":r.get("source_page"),"raw_row_id":r.get("raw_row_id")})

add_issue("duplicate crop_code within year",production[production.duplicated(["year","crop_code"],keep=False)])
add_issue("negative sown/harvested area",production[(production.sown_area_ha<0)|(production.harvested_area_ha<0)],"error")
add_issue("harvested area > sown area",production[(production.harvested_area_ha>production.sown_area_ha+0.01)&production.sown_area_ha.notna()],"warning")
add_issue("post-processing mass > initial mass",production[(production.harvest_after_processing_centners>production.harvest_initial_centners+0.01)&production.harvest_initial_centners.notna()],"warning")
add_issue("positive harvest with missing/non-positive harvested area",production[(production.harvest_tonnes>0)&(~(production.harvested_area_ha>0))],"error")
add_issue("possibly implausible yield > 30 t/ha",production[production.yield_tonnes_per_ha>30],"warning")
qa = pd.DataFrame(issues,columns=["severity","issue","year","crop_code","source_file","source_page","raw_row_id"])
qa.to_csv(OUT / "extraction_qa_issues.csv",index=False)
print("Issues:",len(qa))
display(qa.head(30))
# Sanity check: do not accept an extraction that produced no crop observations.
assert len(production)>20, "Very few crop records: review geometry and pages"
assert set(production.year)==set(range(2021,2026)), "Some years missing from production table"
print("Coverage:",sorted(production.year.unique().tolist()))

Issues: 0


,severity,issue,year,crop_code,source_file,source_page,raw_row_id


Coverage: [2021, 2022, 2023, 2024, 2025]


## 7. Preview & exports

Output files are **created locally** under `data/agriculture/processed/`. Do not commit enterprise-level PDFs or detailed extracts to a public repository without explicit authorization. For the repo, commit this notebook and extraction logic; add local data folders to `.gitignore`.


In [8]:
print("Generated outputs:")
for p in sorted(OUT.glob("*.csv")):
    print(f"  {p.name:32s} {p.stat().st_size / 1024:8.1f} KB")
# Optional spreadsheet preview for review purposes, NOT required for downstream scripts.
# with pd.ExcelWriter(OUT/"agriculture_review.xlsx") as writer:
#     production.to_excel(writer,sheet_name="crop_production",index=False)
#     cost_review.to_excel(writer,sheet_name="50_cost_review",index=False)
#     inventory_review.to_excel(writer,sheet_name="50_stock_review",index=False)


Generated outputs:
  cost_rows_for_review.csv             71.8 KB
  crop_production.csv                 144.8 KB
  extraction_qa_issues.csv              0.1 KB
  inventory_rows_for_review.csv       107.1 KB
  raw_report_rows.csv                1110.8 KB
  report_row_cells.csv                307.0 KB
  source_manifest.csv                   2.1 KB
